# Exercice 6 : Investigation : de l'alerte au rapport

> **En une phrase :** on suit les **7 étapes d'investigation** du cours sur les deux incidents prioritaires de l'exercice 5, et on termine par un rapport d'une page avec des recommandations.

**Le scénario.** Vous avez escaladé deux incidents :
- **A** : 183.62.140.253, la plus grosse force brute de la matinée ;
- **B** : la connexion réussie du compte `fztu` depuis 119.137.62.142.

Votre responsable veut savoir, avant midi : *le serveur est-il compromis ?*

**Les sources.** Le journal SSH (*Loghub*), la géolocalisation d'IP (*ip-api.com*, gratuit) et le renseignement *IPsum* (ex. 3).

**Le plan (les 7 étapes du cours).**
1. Contexte → 2. Enrichissement → 3. Vérification IoC → 4. Chronologie → 5. Hypothèses (MITRE ATT&CK) → 6. Impact → 7. Rapport.

## Étape 0 : exécutez cette cellule en premier
Le notebook est **autonome** : il fonctionne tel quel sur Colab (ouvert depuis Drive) ou en local. Elle doit afficher **Prêt**.

In [1]:
import re, requests
import pandas as pd

URL_SSH = "https://raw.githubusercontent.com/logpai/loghub/master/OpenSSH/OpenSSH_2k.log"

def charger_ssh():
    """Télécharge le journal SSH et le range dans un tableau : une ligne = un événement."""
    lignes = requests.get(URL_SSH, timeout=30).text.splitlines()
    motif = re.compile(r"^(\w{3}\s+\d+ \d\d:\d\d:\d\d) \S+ sshd\[\d+\]: (.*)$")
    evenements = []
    for ligne in lignes:
        m = motif.match(ligne)
        if m:
            ip = re.search(r"\d+\.\d+\.\d+\.\d+", m.group(2))
            evenements.append({"heure": pd.to_datetime("2016 " + m.group(1), format="%Y %b %d %H:%M:%S"),  # l'année n'est pas écrite dans le journal
                               "message": m.group(2),
                               "ip": ip.group() if ip else None})
    return pd.DataFrame(evenements), lignes

evenements, _ = charger_ssh()
ip_A, ip_B = "183.62.140.253", "119.137.62.142"          # <-- MODIFIABLE : les deux incidents à investiguer
print('Prêt')

Prêt


## Étape 1 : Contexte
**Ce qu'on fait :** avant tout, qu'est-ce que ce serveur et qui l'utilise ? On regarde quels comptes se sont **réellement** connectés.

In [2]:
print("Machine :", "LabSZ (un serveur de laboratoire), service SSH ouvert sur Internet")
print("Période :", evenements["heure"].min(), "→", evenements["heure"].max())
reussites = evenements[evenements["message"].str.startswith("Accepted")]
print("Connexions réussies :", len(reussites))
print(reussites[["heure", "message"]].to_string(index=False))

Machine : LabSZ (un serveur de laboratoire), service SSH ouvert sur Internet
Période : 2016-12-10 06:55:46 → 2016-12-10 11:04:45
Connexions réussies : 1
              heure                                                        message
2016-12-10 09:32:20 Accepted password for fztu from 119.137.62.142 port 49116 ssh2


**👀 Ce qu'on doit voir :** **une seule** connexion réussie en 4 heures : `fztu`, par **mot de passe**. C'est donc le seul point d'entrée possible à vérifier.

## Étape 2 : Enrichissement
**Ce qu'on fait :** on ajoute du contexte à chaque IP : pays, fournisseur, numéro de réseau (AS), et s'il s'agit d'un **hébergeur** (serveur loué) ou d'un abonné Internet.

In [3]:
champs = "query,country,city,isp,as,hosting"
geo = requests.post(f"http://ip-api.com/batch?fields={champs}", json=[ip_A, ip_B], timeout=20).json()
enrichi = pd.DataFrame(geo).set_index("query")
enrichi.index = ["A : " + ip_A, "B : " + ip_B]
enrichi

,country,city,isp,as,hosting
A : 183.62.140.253,China,Guangzhou,Chinanet,AS4134 CHINANET BACKBONE,False
B : 119.137.62.142,China,Guangzhou,Chinanet,AS4134 CHINANET BACKBONE,False


**👀 Ce qu'on doit voir :** les **deux** IP sont en **Chine** (Guangzhou), chez le **même fournisseur** (Chinanet, AS4134). Pas des hébergeurs (`hosting = False`) : des connexions d'abonnés. *(La géolocalisation d'aujourd'hui peut différer de celle de l'époque : les adresses sont réattribuées.)*

**❓ Question pour la classe :** même pays, même fournisseur que l'attaquant. Faut-il en conclure que B est l'attaquant ?

*Réponse : non. Chinanet est l'un des plus gros fournisseurs au monde (des centaines de millions d'abonnés) : partager son réseau ne prouve rien. Et le nom du serveur, **LabSZ**, suggère un laboratoire à **Shenzhen**, dans la même province (Guangdong) : un utilisateur légitime local passerait justement par ce fournisseur. L'enrichissement donne des **indices**, jamais une preuve à lui seul.*

## Étape 3 : Vérification IoC
**Ce qu'on fait :** on cherche les deux IP dans le renseignement du jour (IPsum, ex. 3).

In [4]:
texte = requests.get("https://raw.githubusercontent.com/stamparm/ipsum/master/ipsum.txt", timeout=30).text
ipsum = {l.split()[0]: int(l.split()[1]) for l in texte.splitlines() if not l.startswith("#")}
for ip in (ip_A, ip_B):
    print(f"{ip:16} → dans {ipsum.get(ip, 0)} liste(s) noire(s) aujourd'hui")

183.62.140.253   → dans 0 liste(s) noire(s) aujourd'hui
119.137.62.142   → dans 0 liste(s) noire(s) aujourd'hui


**👀 Ce qu'on doit voir :** **0** pour les deux. Comme à l'exercice 3 : le journal est ancien, ces IoC sont **périmés**. Cette étape ne tranche rien ici ; on ne peut compter que sur **nos propres journaux**.

## Étape 4 : Chronologie
**Ce qu'on fait :** on reconstitue, minute par minute, ce qu'a fait chaque IP.

In [5]:
def chronologie(ip):
    e = evenements[evenements["ip"] == ip].copy()
    e["action"] = e["message"].str.extract(r"^(Failed password for (?:invalid user )?\S+|Accepted password for \S+|Invalid user \S+|reverse mapping|Received disconnect|Connection closed)")[0]
    return (e.dropna(subset=["action"])
             .assign(minute=e["heure"].dt.strftime("%H:%M"))
             .groupby(["minute", "action"]).size().rename("fois").reset_index())

print("Incident A :", ip_A)
chrono_A = chronologie(ip_A)
print(chrono_A.head(6).to_string(index=False), "\n   …\n", chrono_A.tail(3).to_string(index=False))
print("\nIncident B :", ip_B)
print(chronologie(ip_B).to_string(index=False))

Incident A : 183.62.140.253
minute                                    action  fois
 10:54      Failed password for invalid user dff     1
 10:54 Failed password for invalid user zhangyan     1
 10:54                  Failed password for root    14
 10:54                          Invalid user dff     1
 10:54                     Invalid user zhangyan     1
 10:54                       Received disconnect    16 
   …
 minute                   action  fois
 11:03      Received disconnect    21
 11:04 Failed password for root    20
 11:04      Received disconnect    20

Incident B : 119.137.62.142
minute                     action  fois
 09:32 Accepted password for fztu     1
 09:45        Received disconnect     1


**👀 Ce qu'on doit voir :** **A** commence à 10:54 et fait **presque uniquement** des `Failed password for root` (≈ 15 à 20 par minute, chacun suivi d'une déconnexion), avec quelques comptes inventés (`dff`, `zhangyan`), jusqu'à la fin du journal (11:04). **B** : une connexion à 09:32, une déconnexion à 09:45. Rien d'autre.

## Étape 5 : Hypothèses (MITRE ATT&CK)
**Ce qu'on fait :** on nomme ce qu'on a vu avec le vocabulaire commun des SOC, **MITRE ATT&CK** : chaque technique a un identifiant que tout analyste comprend.

In [6]:
hypotheses = pd.DataFrame([
    ["A", "Accès aux identifiants", "T1110.001", "Brute Force : Password Guessing", "confirmé (286 essais, compte root)"],
    ["A", "Accès initial",          "T1078",     "Valid Accounts",                   "non observé (aucune réussite)"],
    ["B", "Accès initial",          "T1078",     "Valid Accounts (mot de passe volé ?)", "à vérifier (une réussite, aucun échec)"],
    ["B", "Mouvement latéral",      "T1021.004", "Remote Services : SSH",            "rien dans ce journal"],
], columns=["incident", "tactique", "technique", "nom", "statut"])
hypotheses

,incident,tactique,technique,nom,statut
0,A,Accès aux identifiants,T1110.001,Brute Force : Password Guessing,"confirmé (286 essais, compte root)"
1,A,Accès initial,T1078,Valid Accounts,non observé (aucune réussite)
2,B,Accès initial,T1078,Valid Accounts (mot de passe volé ?),"à vérifier (une réussite, aucun échec)"
3,B,Mouvement latéral,T1021.004,Remote Services : SSH,rien dans ce journal


**❓ Question pour la classe :** pour B, l'hypothèse « mot de passe volé » (T1078) ne peut pas être écartée avec ce journal seul. Quelles autres sources demanderiez-vous ?

*Réponse : (1) demander à **l'utilisateur fztu** s'il s'est connecté à 09:32 ; (2) son **historique** : se connecte-t-il habituellement depuis cette IP, à cette heure ? (UEBA, détection d'anomalies sur l'utilisateur) ; (3) les **journaux de commandes** de la session (`.bash_history`, `auditd`) ; (4) vérifier si son mot de passe figure dans une **fuite** (Ch. 5, ex. 1). C'est la corrélation multi-source de l'exercice 5.*

## Étape 6 : Impact
**Ce qu'on fait :** quels comptes ont été touchés ? Un attaquant a-t-il réussi **quoi que ce soit** ?

In [7]:
echecs = evenements[evenements["message"].str.startswith("Failed password")]
ip_attaquantes = set(echecs["ip"])
reussites = evenements[evenements["message"].str.startswith("Accepted")]

print("IP ayant échoué au moins une fois :", len(ip_attaquantes))
print("Parmi elles, IP ayant ensuite RÉUSSI :", len(ip_attaquantes & set(reussites["ip"])))
print("Connexions réussies sur root :", reussites["message"].str.contains("for root").sum())
print("Comptes essayés par les attaquants :", echecs["message"].str.extract(r"for (?:invalid user )?(\S+) from")[0].nunique())

IP ayant échoué au moins une fois : 23
Parmi elles, IP ayant ensuite RÉUSSI : 0
Connexions réussies sur root : 0
Comptes essayés par les attaquants : 62


**👀 Ce qu'on doit voir :** **23** IP attaquantes, **0** a réussi, **0** connexion root. Les attaquants ont essayé **62** comptes différents, sans succès.

## Étape 7 : Rapport
**Ce qu'on fait :** une page, lisible par un gestionnaire : ce qui s'est passé, ce qu'on sait, ce qu'on ne sait pas, ce qu'il faut faire.

In [8]:
nb_A = (echecs["ip"] == ip_A).sum()
print(f"""RAPPORT D'INVESTIGATION : serveur LabSZ (SSH)
{'=' * 55}
Période analysée : {evenements['heure'].min():%H:%M} → {evenements['heure'].max():%H:%M} ({len(evenements)} événements)

CONSTAT
- {len(echecs)} échecs de connexion venant de {len(ip_attaquantes)} IP : attaques automatisées
  par dictionnaire (MITRE T1110.001), visant surtout le compte root.
- Incident A ({ip_A}) : {nb_A} essais en 10 min. ÉCHEC, aucun accès.
- Incident B ({ip_B}) : 1 connexion réussie du compte fztu, sans échec préalable.
  Probablement légitime. NON CONFIRMÉ.

CONCLUSION : aucune compromission observée dans ce journal.
             Le doute sur B doit être levé avant de fermer le dossier.

RECOMMANDATIONS
1. Confirmer la connexion de 09:32 avec l'utilisateur fztu.
2. Interdire la connexion SSH en root (PermitRootLogin no).
3. Remplacer les mots de passe SSH par des clés, ou ajouter le MFA (Ch. 4).
4. Bloquer automatiquement après 5 échecs en 2 min (fail2ban), selon la règle de l'ex. 2.
5. Ne plus exposer SSH à tout Internet : VPN ou liste d'IP autorisées.""")

RAPPORT D'INVESTIGATION : serveur LabSZ (SSH)
Période analysée : 06:55 → 11:04 (2000 événements)

CONSTAT
- 518 échecs de connexion venant de 23 IP : attaques automatisées
  par dictionnaire (MITRE T1110.001), visant surtout le compte root.
- Incident A (183.62.140.253) : 286 essais en 10 min. ÉCHEC, aucun accès.
- Incident B (119.137.62.142) : 1 connexion réussie du compte fztu, sans échec préalable.
  Probablement légitime. NON CONFIRMÉ.

CONCLUSION : aucune compromission observée dans ce journal.
             Le doute sur B doit être levé avant de fermer le dossier.

RECOMMANDATIONS
1. Confirmer la connexion de 09:32 avec l'utilisateur fztu.
2. Interdire la connexion SSH en root (PermitRootLogin no).
3. Remplacer les mots de passe SSH par des clés, ou ajouter le MFA (Ch. 4).
4. Bloquer automatiquement après 5 échecs en 2 min (fail2ban), selon la règle de l'ex. 2.
5. Ne plus exposer SSH à tout Internet : VPN ou liste d'IP autorisées.


**👀 Ce qu'on doit voir :** un rapport d'une quinzaine de lignes : le constat chiffré, une conclusion **prudente** (« aucune compromission observée », pas « aucune compromission »), et **cinq actions** concrètes.

## Ce qu'il faut retenir
- L'investigation suit une méthode : **contexte, enrichissement, IoC, chronologie, hypothèses, impact, rapport**. Sauter une étape, c'est risquer une mauvaise conclusion.
- L'enrichissement et les IoC donnent des **indices** ; seuls **nos journaux** donnent des faits. Ici, le renseignement public n'a rien apporté (IoC périmés).
- **MITRE ATT&CK** donne un vocabulaire commun pour décrire ce que fait l'attaquant.
- Un bon rapport distingue **ce qu'on sait** de **ce qu'on suppose**, et finit par des **actions**. Plusieurs d'entre elles (MFA, clés, pas de root) rendent la force brute **inutile**, au lieu de simplement la détecter.

**Pour aller plus loin :** refaites les exercices 1 à 6 sur le journal complet de ce serveur (655 146 lignes sur 28 jours, 70 Mo : https://zenodo.org/records/8196385/files/SSH.tar.gz). Les règles tiennent-elles encore ? Combien d'alertes, combien d'incidents ?